# Import libraries

In [1]:
import numpy as np
import pandas as pd
import regex as re
import langid
from langdetect import detect_langs
import nltk
import string

# Import data

In [2]:
df_spotify = pd.read_csv("Data/spotify_songs_original.csv")

In [3]:
df_deezer = pd.read_csv("Data/deezer_songs_original.csv")

In [14]:
df_spotify_old = pd.read_csv('Data/spotify_songs_with_translation_information.csv')
df_deezer_old  = pd.read_csv('Data/deezer_songs_with_translation_information.csv')

In [73]:
nltk_data_dir = '/Data/Nltk/'
nltk.data.path.append(nltk_data_dir)
nltk.download('words', download_dir=nltk_data_dir)

[nltk_data] Downloading package words to /Data/Nltk/...
[nltk_data]   Package words is already up-to-date!


True

In [74]:
# English vocabulary set (lowercased for fast lookup)
EN_VOCAB = set(w.lower() for w in nltk.corpus.words.words())
#Some RAP music were flag as non English without adding English word which were not in the corpus' words, we also add musical artefact
EN_VOCAB.update({'nigga', 'fuck', 'tv', 'shit', 'mama', 'ay', 'ayy', 'dr', 'kid', 'children', 'na', 'oo', 'ooh', 'mmm', 'vibe', 'dawg', 'trippin', 'tryna', 'sittin', 'gettin'}) 

# Functions

In [67]:
def detect_to_translate_langid(text):
    lang, _ = langid.classify(text)
    if lang == 'en':
        return False
    return True

In [68]:
def eng_confidence(text: str) -> float:
    if not text or text.strip() == "":
        return 0.0
    lang, conf = langid.classify(text)
    if lang == 'en':
        return conf
    return 0.0

In [69]:
def has_non_latin(text):
    # Matches any character *outside* basic Latin + Latin-1 supplement
    return bool(re.search(r'[^\u0000-\u024F]', text))

def is_english_word(word):
    if has_non_latin(word):
        return False
    if word[0].isupper():
        return True
    try:
        int(word)
        return True
    except:
        pass
    w = re.sub(r'[^a-zA-Z]', '', word).lower()
    if not w:
        return False
    if w in EN_VOCAB:
        return True
    if w[-1] == 's':
        if w[:-1] in EN_VOCAB:
            return True
    if (w[-2:] == 'er' or w[-2:] == 'ed'):
        if w[:-2] in EN_VOCAB:
            return True
    if w[-2:] == 'in':
        if (w[:-2] in EN_VOCAB or w[:-2]+'e' in EN_VOCAB):
            return True
    if w[-3:] == 'ing':
        if (w[:-3] in EN_VOCAB or w[:-3]+'e' in EN_VOCAB):
            return True
    return False

def remove_punctuation(text):
    """
    Removes all punctuation from the text (keeps letters, digits, and spaces).
    Works safely for multilingual Unicode text.
    """
    # Replace all punctuation characters with space
    text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text)
    # Also remove other Unicode punctuation/symbols
    text = re.sub(r"[\u2000-\u206F\u2E00-\u2E7F\'\"“”‘’«»…•·–—−]", " ", text)
    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

def english_ratio(segment, verbose = False):
    segment = remove_punctuation(segment)
    tokens = re.findall(r'\b\w+\b', segment)
    if not tokens:
        return 0.0
    if verbose:
        for t in tokens:
            if not is_english_word(t):
                print(t)
    eng_count = sum(is_english_word(t) for t in tokens)
    return eng_count / len(tokens)

In [70]:
def detect_lang_dual(text):
    """Use both langdetect and langid."""
    try:
        l1 = detect_langs(text)[0].lang
        l2, score = langid.classify(text)
        if (l1 == 'en') and (l2 == 'en'):
            return 'en'
    except:
        pass
    return 'other'

def split_sentences(text):
    """Split lyric into sentence-like fragments for better language context."""
    return re.split(r'(?<=[\.\?!])\s+', text)

In [71]:
def detect_song_language(lyric):
    """Detect if the song is purely English or mixed."""
    sentences = split_sentences(lyric)
    langs = [detect_lang_dual(s) for s in sentences if s.strip()]
    english_ratio = sum(l == 'en' for l in langs) / len(langs)
    return 'en' if english_ratio >= 0.8 else 'mixed'

# Detecting if the lyrics might need translation

## Spotify dataset

In [ ]:
df_spotify['lyric_length'] = df_spotify.lyrics.apply(lambda x : len(x.split(' ')))
df_spotify['english_word_ratio'] = df_spotify.lyrics.apply(english_ratio)

In [ ]:
df_spotify['is_english_only_ratio'] = df_spotify.english_word_ratio <= 0.85
df_spotify['is_only_english_detect'] = df_spotify.lyrics.apply(detect_song_language) != 'en'

Both methods agreeing in most cases proves their respective pertinence

In [ ]:
(df_spotify['is_only_english_detect'] == df_spotify['is_english_only_ratio']).mean()

np.float64(0.9854040332816246)

To be sure, we consider the lyric needs translation if any of the two methods has a doubt concerning the lyric being written in English.

In [ ]:
df_spotify['needs_translation'] = (df_spotify['is_only_english_detect'] + df_spotify['is_english_only_ratio']) > 0.5

In [ ]:
df_spotify.drop(['is_english_only_ratio', 'is_only_english_detect'], axis = 1).to_csv('Data/spotify_songs_with_translation_information.csv')

## Deezer dataset

In [75]:
df_deezer['lyric_length'] = df_deezer.lyrics.apply(lambda x : len(x.split(' ')))
df_deezer['english_word_ratio'] = df_deezer.lyrics.apply(english_ratio)

In [76]:
df_deezer['is_english_only_ratio'] = df_deezer.english_word_ratio >= 0.85
df_deezer['is_only_english_detect'] = df_deezer.lyrics.apply(detect_song_language) == 'en'

In [78]:
(df_deezer['is_only_english_detect'] == df_deezer['is_english_only_ratio']).mean()

np.float64(0.9346171070309002)

In [79]:
df_deezer['needs_translation'] = (df_deezer['is_only_english_detect'] + df_deezer['is_english_only_ratio']) > 0.5

In [ ]:
df_deezer.drop(['is_english_only_ratio', 'is_only_english_detect'], axis = 1).to_csv('Dataset/deezer_songs_with_translation_information.csv')